# MJX 09 — Playground PPO: Franka Panda pick-and-place

### Lab Description

The capstone manipulation task: **`PandaPickCube`** — a 7-DoF Franka Panda arm that must reach, grasp, and lift a cube. This is a real robot arm from the MuJoCo Menagerie, so it is far harder (and far easier to destabilise) than the DM Control toy tasks.

**Why this notebook is tuned differently.** With the stock high learning rate, PPO on this task **diverges to NaN** on the gfx1151 APU. We trade speed for stability — learning is slower but never blows up — using a lower learning rate, gradient clipping, observation normalization, and reward scaling. The goal here is a *stable* curve with no NaNs, even if it climbs gradually.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Train PPO on a realistic 7-DoF manipulation task
- Apply stabilization (low LR, gradient clipping, obs normalization, reward scaling) to prevent NaNs
- Render the Panda attempting the pick-and-place

### Import libraries, shim, and ensure robot assets

Besides the usual shim and matmul-precision setting, the Panda model and its meshes come from the **MuJoCo Menagerie**. The course image pre-downloads them at build time; the `ensure_menagerie_exists()` call here is a no-op safety net.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground._src import mjx_env
from mujoco_playground.config import manipulation_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

# The Panda model + assets live in the MuJoCo Menagerie. The image pre-downloads
# them at build time; this is a no-op safety net if they are already present.
try:
    mjx_env.ensure_menagerie_exists()
except Exception as e:
    print("menagerie check:", e)

print("JAX devices:", jax.devices())

### Load PandaPickCube and build a *stabilized* PPO config

We start from Playground's manipulation config, then apply the stability-first overrides: **`learning_rate=5e-5`** (well below default), **`max_grad_norm=1.0`** (gradient clipping), **`normalize_observations=True`**, and **`reward_scaling=0.1`**. These together stop the early NaN divergence seen on this APU.

In [ ]:
ENV_NAME = "PandaPickCube"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

cfg = manipulation_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
# Stability-first overrides (see notebook header).
ppo_params.update(
    # SHORT demo run (~5 min): brax rounds num_timesteps up to whole eval
    # intervals (~2.46M steps each); 2 intervals = 4,915,200 steps. That is
    # enough to SEE the reward climbing (to a few hundred) but NOT enough to
    # solve the task. A reliable grasp-and-lift (eval reward ~1300+) needs
    # roughly 15-25M steps (~20-30 min on this APU) — so instead of waiting, we
    # ship a fully-trained policy and load it below (checkpoints/panda512_final.pkl).
    num_timesteps=4_915_200,
    num_envs=512,        # 512 is the stable ceiling here; 1024+ NaNs / collapses on this APU
    batch_size=512,
    num_minibatches=8,
    num_evals=3,
    learning_rate=1e-4,  # low + grad-clipped to avoid the NaN divergence this task is prone to
    max_grad_norm=1.0,
    normalize_observations=True,
    reward_scaling=0.1,
    clipping_epsilon=0.2,
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "learning_rate", "max_grad_norm"]})

### Train (short ~5-minute demo)

Run PPO on the GPU. This is a deliberately **short run (~5 min, ≈5M steps)** so
you can watch the eval reward *start to climb* (from ~100 up to a few hundred)
without waiting for full convergence. `progress_fn` flags any NaN explicitly —
with the stabilized config you should see a steady climb instead.

**How long does a full solve take?** Reaching a reliable grasp-and-lift
(eval reward ~1300-1400) takes roughly **15-25M steps (~20-30 minutes)** on this
APU — the reward jumps once the policy discovers how to close the gripper and
lift. Rather than wait, the next section loads a checkpoint already trained that
far, so the render shows a successful pick.

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    flag = "  <-- NaN!" if np.isnan(r) else ""
    print(f"step {int(step):>9}  eval reward {r:8.2f}{flag}")

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    seed=0,
)
print("training done")

### Plot the learning curve

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME}) — stabilised"); plt.grid(True); plt.show()

### Render the half-trained policy (it can't grasp yet)

This is the policy from the short ~5-minute run above. The reward was climbing
but the arm has only learned to *approach* the cube — it does not yet close the
gripper and lift. Watch it fall short, then compare with the fully-trained
checkpoint in the next section.

In [ ]:
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

# Hide the translucent target marker (the faint red "ghost" cube).
_rgba = np.asarray(env.mj_model.geom_rgba)
env.mj_model.geom_rgba[_rgba[:, 3] < 1.0, 3] = 0.0

rng = jax.random.PRNGKey(1)
state = reset(rng)
trajectory = [state]
for _ in range(150):
    rng, k = jax.random.split(rng)
    action, _ = inference(state.obs, k)
    state = step(state, action)
    trajectory.append(state)

frames = np.asarray(env.render(trajectory, height=240, width=320))
imageio.mimsave("output/videos/mjx09_after_short_train.mp4", list(frames), fps=30)
print("saved", frames.shape[0], "frames (half-trained policy)")

In [ ]:
Video(url="output/videos/mjx09_after_short_train.mp4")

### Load the fully-trained checkpoint

The short run above only got the arm reaching, not grasping. This course ships a
policy trained all the way to convergence at `checkpoints/panda512_final.pkl`
(final eval reward ≈ 1400 — a reliable grasp-and-lift, ~20-30 min of training).
Load it so the render below shows a successful pick.

In [ ]:
from brax.io import model

# Rebuild the policy network without training (num_timesteps=0 just constructs
# it), then load the pretrained weights shipped with the course.
build_fn = functools.partial(ppo.train, **{**ppo_params, "num_timesteps": 0})
if net_cfg:
    build_fn = functools.partial(build_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))
make_inference_fn, _, _ = build_fn(environment=env,
                                   wrap_env_fn=wrapper.wrap_for_brax_training, seed=0)
params = model.load_params("checkpoints/panda512_final.pkl")
print("loaded pretrained checkpoint (eval reward ~1400, reliable grasp)")

### Render the pretrained policy — a successful pick

Now the loaded checkpoint drives the arm: it reaches the cube, closes the
gripper, and lifts it off the table.

In [ ]:
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

# Hide the translucent target marker (the faint red "ghost" cube that shows
# where the block should be placed) so only the real cube and arm are rendered.
_rgba = np.asarray(env.mj_model.geom_rgba)
env.mj_model.geom_rgba[_rgba[:, 3] < 1.0, 3] = 0.0

rng = jax.random.PRNGKey(1)
state = reset(rng)
trajectory = [state]
for _ in range(150):
    rng, k = jax.random.split(rng)
    action, _ = inference(state.obs, k)
    state = step(state, action)
    trajectory.append(state)

frames = np.asarray(env.render(trajectory, height=240, width=320))
out = "output/videos/mjx09_panda_pick_cube.mp4"
imageio.mimsave(out, list(frames), fps=30)
print("saved", frames.shape[0], "frames ->", out)

### Watch the result

In [ ]:
Video(url="output/videos/mjx09_panda_pick_cube.mp4")

## Conclusions

You trained a real 7-DoF Panda arm with PPO on the AMD GPU and, crucially, learned how to *stabilize* a fragile manipulation run (lower learning rate + gradient clipping + normalization) to avoid the NaN divergence that the stock hyperparameters cause on this hardware. This completes the MuJoCo MJX course: from MuJoCo basics, through GPU-parallel MJX, to Playground PPO on toy control and a real robot arm.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT